In [1]:
"""
IMDb bulk-dataset ingestion -- dumps title.ratings.tsv.gz and
title.basics.tsv.gz into a temporary imdb.db staging database.

Same shape as tmdb_popularity: per file, download -> stream-read ->
store -> delete, before moving on. Nothing large is ever held in
memory: the .gz is streamed to disk in chunks, read back line by line,
and written in bounded batches.

By default only rows whose tconst is already an imdb_id in your
cinesync `titles` table are kept (pass keep_ids=None to dump every
row instead -- note title.basics is ~11M rows / >1GB uncompressed).

Source: https://datasets.imdbws.com/  (personal / non-commercial use)
Fields are '\\N' for null; ratings/votes/years/runtime are cast to
numeric, everything else stored verbatim. `genres` stays the raw
comma-joined string ("Action,Comedy,Crime") -- splitting into
cinesync's title_genres belongs to the later merge step, not here.
"""

import gzip
import sqlite3
from pathlib import Path

import requests

from cinesync.paths import DATA_DIR, DB_PATH

BASE_URL = "https://datasets.imdbws.com/"
TMP_DIR = DATA_DIR / "imdb_bulk"

IMDB_SCHEMA = """
CREATE TABLE IF NOT EXISTS imdb_ratings (
    tconst          TEXT NOT NULL,
    average_rating  REAL,
    num_votes       INTEGER,
    PRIMARY KEY (tconst)
);

CREATE TABLE IF NOT EXISTS imdb_basics (
    tconst           TEXT NOT NULL,
    title_type       TEXT,
    primary_title    TEXT,
    original_title   TEXT,
    is_adult         INTEGER,
    start_year       INTEGER,
    end_year         INTEGER,
    runtime_minutes  INTEGER,
    genres           TEXT,
    PRIMARY KEY (tconst)
);
"""

# Per file: which IMDb header column maps to which db column, and how to cast
# it. (imdb_header_name, db_column, cast). `str` casts are stored verbatim.
RATINGS_SPEC = {
    "filename": "title.ratings.tsv.gz",
    "table": "imdb_ratings",
    "columns": [
        ("tconst", "tconst", str),
        ("averageRating", "average_rating", float),
        ("numVotes", "num_votes", int),
    ],
}

BASICS_SPEC = {
    "filename": "title.basics.tsv.gz",
    "table": "imdb_basics",
    "columns": [
        ("tconst", "tconst", str),
        ("titleType", "title_type", str),
        ("primaryTitle", "primary_title", str),
        ("originalTitle", "original_title", str),
        ("isAdult", "is_adult", int),
        ("startYear", "start_year", int),
        ("endYear", "end_year", int),
        ("runtimeMinutes", "runtime_minutes", int),
        ("genres", "genres", str),
    ],
}


def create_imdb_db(path):
    """Open (creating if needed) the staging DB and ensure the schema exists.
    Idempotent -- safe to call on every run."""
    conn = sqlite3.connect(path)
    conn.executescript(IMDB_SCHEMA)
    conn.commit()
    return conn


def load_known_imdb_ids(cinesync_db_path):
    """The set of tconsts to keep = every imdb_id in cinesync's titles table."""
    with sqlite3.connect(cinesync_db_path) as cs:
        rows = cs.execute(
            "SELECT DISTINCT imdb_id FROM titles "
            "WHERE imdb_id IS NOT NULL AND imdb_id != ''"
        ).fetchall()
    return {r[0] for r in rows}


def _cast(raw, cast):
    """'\\N' / empty -> None; otherwise cast, tolerating the odd malformed value."""
    if raw == "\\N" or raw == "":
        return None
    if cast is str:
        return raw
    try:
        return cast(raw)
    except (ValueError, TypeError):
        return None


def download_dataset(filename, session, tmp_dir=TMP_DIR, timeout=60):
    """Stream a .gz file to disk in 1MB chunks -- never buffers the whole file."""
    tmp_dir.mkdir(parents=True, exist_ok=True)
    tmp_path = tmp_dir / filename
    url = BASE_URL + filename
    with session.get(url, stream=True, timeout=timeout) as resp:
        resp.raise_for_status()
        with open(tmp_path, "wb") as fh:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                fh.write(chunk)
    return tmp_path


def ingest_tsv_file(conn, gz_path, spec, keep_ids=None, batch_size=50_000):
    """
    Stream a gzipped IMDb TSV line by line and upsert into spec['table'].

    Header row drives column positions (robust to IMDb adding columns).
    Rows are written in bounded batches so neither read nor write side
    holds the file in memory. INSERT OR REPLACE (keyed on tconst) makes
    re-runs refresh values rather than duplicate -- ratings/votes drift
    over time. Returns (written, skipped_malformed).
    """
    cols = spec["columns"]
    db_cols = [c[1] for c in cols]
    placeholders = ", ".join(["?"] * len(db_cols))
    sql = (
        f"INSERT OR REPLACE INTO {spec['table']} ({', '.join(db_cols)}) "
        f"VALUES ({placeholders})"
    )

    written = skipped = 0
    batch = []
    with gzip.open(gz_path, "rt", encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split("\t")
        pos = {name: i for i, name in enumerate(header)}
        idx = [(pos[src], cast) for src, _db, cast in cols]  # KeyError if IMDb drops a col
        tconst_i = pos["tconst"]
        n_expected = len(header)

        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) != n_expected:
                skipped += 1
                continue
            if keep_ids is not None and fields[tconst_i] not in keep_ids:
                continue
            batch.append([_cast(fields[i], cast) for i, cast in idx])
            if len(batch) >= batch_size:
                conn.executemany(sql, batch)
                conn.commit()
                written += len(batch)
                batch.clear()

        if batch:
            conn.executemany(sql, batch)
            conn.commit()
            written += len(batch)

    return written, skipped


def process_one_dataset(conn, spec, session, keep_ids):
    """Download -> ingest -> delete, in that order. Delete runs even on failure."""
    gz_path = download_dataset(spec["filename"], session)
    try:
        return ingest_tsv_file(conn, gz_path, spec, keep_ids)
    finally:
        gz_path.unlink(missing_ok=True)


def run_ingestion(conn, keep_ids=None, specs=(RATINGS_SPEC, BASICS_SPEC)):
    """
    keep_ids: set of tconsts to keep (typically load_known_imdb_ids(cinesync.db)),
              or None to store every row in the file.
    """
    session = requests.Session()
    for spec in specs:
        print(f"Downloading {spec['filename']} ...")
        written, skipped = process_one_dataset(conn, spec, session, keep_ids)
        msg = f"  {spec['table']}: {written} rows written"
        if skipped:
            msg += f", {skipped} malformed lines skipped"
        print(msg)

In [2]:
conn = create_imdb_db(DATA_DIR / "imdb.db")
keep = load_known_imdb_ids(DB_PATH)
run_ingestion(conn, keep_ids=keep) 
conn.close()

  imdb_ratings: 70854 rows written
  imdb_basics: 70981 rows written
